In [ ]:
from datasets import UBFC_Dataset
import os
from torch.utils.data import DataLoader 
from torch.utils.data import random_split 
import torch
data_path = os.path.join('../data','UBFC-RPPG-Dataset')
subjects = os.listdir(data_path)  
dataset = UBFC_Dataset(data_path, subjects) 

In [ ]:
class rPPGModel(torch.nn.Module):
    def __init__(self,input_size=9, d_model=64, nhead=4, ff_hidden_size=128, num_layers=2, output_size=1):
        super(rPPGModel, self).__init__()
        self.input_size = input_size
        self.d_model = d_model
        self.nhead = nhead
        self.ff_hidden_size = ff_hidden_size
        self.num_layers = num_layers
        self.output_size = output_size


        self.encoder_embedding = torch.nn.Linear(self.input_size, self.d_model)
        self.transformer_encoder_layer = torch.nn.TransformerEncoderLayer(d_model=self.d_model, nhead=self.nhead, dim_feedforward=self.ff_hidden_size,batch_first=True,activation="gelu")
        self.transformer_encoder = torch.nn.TransformerEncoder(self.transformer_encoder_layer, num_layers=self.num_layers)

        self.decoder_embedding = torch.nn.Linear(self.output_size, self.d_model)
        self.transformer_decoder_layer = torch.nn.TransformerDecoderLayer(d_model=self.d_model, nhead=self.nhead, dim_feedforward=self.ff_hidden_size,batch_first=True,activation="gelu")
        self.transformer_decoder = torch.nn.TransformerDecoder(self.transformer_decoder_layer, num_layers=self.num_layers)


        self.head = torch.nn.Linear(self.d_model, self.output_size)

    def forward(self, src, tgt):
        #src: (B, T, input_size) , target: (B, T, output_size)
        
        src = self.encoder_embedding(src) #(B, T, d_model)
        memory = self.transformer_encoder(src) #(B, T, d_model)

        tgt = self.decoder_embedding(tgt) #(B, T, d_model)
        mask = torch.triu(torch.ones(tgt.size(1), tgt.size(1), device=tgt.device), diagonal=1).bool() #(T, T)

        decoder_output = self.transformer_decoder(tgt, memory  , tgt_mask=mask) #(B, T, d_model)
        output = self.head(decoder_output) #(B, T, output_size)
        return output

        
    

In [ ]:
class MSE_NegPearsonLoss(torch.nn.Module):
    def __init__(self):
        super(MSE_NegPearsonLoss, self).__init__()
        self.mse_loss = torch.nn.MSELoss()
    def forward(self,preds, targs):

        preds = preds.flatten()
        targs = targs.flatten()
        preds_mean = torch.mean(preds)
        targs_mean = torch.mean(targs)
#(B,T,1)
        preds_std = torch.std(preds)
        targs_std = torch.std(targs)
        
        z_preds = (preds - preds_mean )/preds_std 
        z_targs = (targs - targs_mean) / targs_std 

        pearson_corr = torch.mean(z_preds * z_targs)

        mse_loss = self.mse_loss(preds,targs) 

        return  mse_loss , 1-pearson_corr
    


model = rPPGModel(input_size=9, d_model=64, nhead=4, ff_hidden_size=128, num_layers=2, output_size=1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
random_seed = 42
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(random_seed))

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



def evaluate():
    model.eval()
    total_loss = 0 
    for color_seq, signal_seq in test_loader:
            preds = model(color_seq, signal_seq)
            mse_loss,neg_pearson_loss = MSE_NegPearsonLoss()(preds, signal_seq)
            curr_loss = mse_loss + neg_pearson_loss
            total_loss += curr_loss.item()
    

    
    avg_loss =  total_loss / len(test_dataset)
    model.train()
    return avg_loss


            
epochs = 10 
log_every = 10
best_loss = float('inf')
if os.path.exists("best_model.pth"):
     chkpt = torch.load("best_model.pth")
     model.load_state_dict(chkpt["model_state_dict"])
     optimizer.load_state_dict(chkpt["optimizer_state_dict"])
     best_loss = chkpt["loss"]
    

for epoch in range(epochs):
    model.train()
    for idx, (color_seq, signal_seq) in enumerate(train_loader):
        optimizer.zero_grad()
        preds = model(color_seq, signal_seq)
        mse_loss,neg_pearson_loss = MSE_NegPearsonLoss()(preds, signal_seq)
        loss = mse_loss + neg_pearson_loss
        loss.backward()
        optimizer.step()

        if (idx + 1) % log_every == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{idx+1}/{len(train_loader)}], Mse loss: {mse_loss.item():.4f}, Neg Pearson loss: {neg_pearson_loss.item():.4f}, Total loss: {loss.item():.4f}")
    val_loss = evaluate()
    chkpt = {
         "epoch":epoch,
         "loss":loss.item(),
         "optimizer_state_dict":optimizer.state_dict(),
         "model_state_dict":model.state_dict()

    }
    if val_loss < best_loss:
        print(f"New Best Loss: {val_loss}")
        torch.save(chkpt, "best_model.pth")
    else:
         torch.save(chkpt, "latest.pth")    
       

Epoch [1/10], Step [1/430], mse_loss: 0.6557, neg_pearson_loss: 0.1407, total_loss: 0.7964
Epoch [1/10], Step [2/430], mse_loss: 0.7713, neg_pearson_loss: 0.0657, total_loss: 0.8370
Epoch [1/10], Step [3/430], mse_loss: 0.2120, neg_pearson_loss: 0.0430, total_loss: 0.2550
Epoch [1/10], Step [4/430], mse_loss: 0.4388, neg_pearson_loss: 0.0372, total_loss: 0.4760
Epoch [1/10], Step [5/430], mse_loss: 0.1722, neg_pearson_loss: 0.0343, total_loss: 0.2064
Epoch [1/10], Step [6/430], mse_loss: 0.0879, neg_pearson_loss: 0.0397, total_loss: 0.1276
Epoch [1/10], Step [7/430], mse_loss: 0.2129, neg_pearson_loss: 0.0405, total_loss: 0.2534
Epoch [1/10], Step [8/430], mse_loss: 0.2075, neg_pearson_loss: 0.0359, total_loss: 0.2434
Epoch [1/10], Step [9/430], mse_loss: 0.1061, neg_pearson_loss: 0.0334, total_loss: 0.1394
Epoch [1/10], Step [10/430], mse_loss: 0.0667, neg_pearson_loss: 0.0291, total_loss: 0.0958
Epoch [1/10], Step [11/430], mse_loss: 0.1054, neg_pearson_loss: 0.0251, total_loss: 0.13

KeyboardInterrupt: 